# 🤖 Building a RAG Chatbot with LangChain — Full Tutorial

**Corso di Programmazione — Prof.ssa Flora Amato**  
**Università degli Studi di Napoli Federico II — DIETI**

---

## 📌 What You Will Learn

In this notebook, we will build a **Retrieval-Augmented Generation (RAG)** chatbot step by step.  
By the end, you will understand:

1. **What RAG is** and why it matters
2. How to **load and split documents** into chunks
3. How to create **vector embeddings** of text
4. How to store and search vectors in a **vector store**
5. How to connect a **Large Language Model (LLM)** to retrieved context
6. How to build a **conversational chain** with memory
7. How to create a simple **chat UI** inside Colab
8. How to **upload your own files** and chat with them
9. How to **evaluate** your RAG pipeline with **RAGAS**
10. How to build a **Gradio web app** for your chatbot

---

## 📖 Part 0 — What is RAG?

### The Problem
Large Language Models (LLMs) like GPT-4 or Claude are trained on vast amounts of data, but they have **two critical limitations**:

| Limitation | Example |
|---|---|
| **Knowledge cutoff** | The model doesn't know about events after its training date |
| **No access to private data** | The model can't read your company documents, PDFs, or databases |

### The Solution: RAG

**Retrieval-Augmented Generation** solves this by adding a *retrieval step* before generation:

```
User Question
     │
     ▼
┌─────────────┐
│  RETRIEVER   │ ──► Search your documents for relevant passages
└─────────────┘
     │
     ▼ (relevant context)
┌─────────────┐
│  GENERATOR   │ ──► LLM generates answer USING the retrieved context
│   (LLM)      │
└─────────────┘
     │
     ▼
  Answer grounded in YOUR data
```

### Why is this powerful?
- ✅ The LLM answers based on **your actual documents**
- ✅ Reduces **hallucinations** (making things up)
- ✅ No need to **retrain** the model
- ✅ Works with **any document type** (PDF, web pages, databases, etc.)

---

## 🏗️ Architecture Overview

```
                        ┌──────────────────────────────────────┐
                        │        INDEXING PHASE (offline)       │
                        │                                      │
  Documents ──► Load ──► Split into Chunks ──► Embed ──► Store │
  (PDF, txt,    (1)         (2)                 (3)     in DB  │
   web, ...)                                            (4)    │
                        └──────────────────────────────────────┘

                        ┌──────────────────────────────────────┐
                        │        QUERY PHASE (online)          │
                        │                                      │
  User Query ──► Embed ──► Search Vector DB ──► Retrieve Top-K │
     (5)         (6)           (7)               Chunks (8)    │
                        │                                      │
                        │  Prompt + Context ──► LLM ──► Answer │
                        │       (9)            (10)     (11)   │
                        └──────────────────────────────────────┘
```

---

## ⚙️ Part 1 — Setup & Installation

| Library | Purpose |
|---|---|
| `langchain` | Core framework for building LLM applications |
| `langchain-community` | Community integrations (loaders, vector stores, etc.) |
| `langchain-openai` | OpenAI-specific LLM and embedding wrappers |
| `langchain-huggingface` | HuggingFace embeddings (free, no API key needed) |
| `chromadb` | Lightweight vector database (runs locally) |
| `pypdf` | For reading PDF files |
| `ragas` | Framework for evaluating RAG pipelines |
| `gradio` | Build web UIs for ML models |

In [ ]:
#@title 📦 Install All Required Packages

!pip install -q langchain langchain-community langchain-openai langchain-huggingface
!pip install -q chromadb pypdf tiktoken
!pip install -q sentence-transformers
!pip install -q ragas datasets
!pip install -q gradio

print("\n✅ All packages installed successfully!")

In [ ]:
#@title 🔑 API Key Configuration

import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("🔑 Enter your OpenAI API Key (or press Enter to skip): ")

USE_OPENAI = bool(os.environ.get("OPENAI_API_KEY", "").strip())

if USE_OPENAI:
    print("✅ OpenAI API key set. We will use GPT-3.5-turbo as the LLM.")
else:
    print("⚠️ No OpenAI key provided. We will use HuggingFace (free) as a fallback.")
    print("   Note: Free models are slower and less capable, but great for learning!")

---
## 📄 Part 2 — Loading Documents

LangChain provides **Document Loaders** for many formats. Each loader returns `Document` objects with:
- `page_content` → the actual text
- `metadata` → information about the source (filename, page number, etc.)

In [ ]:
#@title 📝 Create Sample Documents

import os
os.makedirs("sample_docs", exist_ok=True)

with open("sample_docs/python_basics.txt", "w") as f:
    f.write("""Python Programming Language — A Comprehensive Overview

Python is a high-level, interpreted programming language created by Guido van Rossum
and first released in 1991. It emphasizes code readability with its notable use of
significant indentation. Python supports multiple programming paradigms, including
structured, object-oriented, and functional programming.

Key Features of Python:
Python uses dynamic typing and garbage collection. It has a large standard library
that is often described as having a "batteries included" philosophy. The language
provides constructs that enable clear programming on both small and large scales.

Data Types in Python:
Python has several built-in data types: integers (int), floating-point numbers (float),
strings (str), booleans (bool), lists, tuples, dictionaries (dict), and sets.
Lists are mutable ordered sequences, while tuples are immutable. Dictionaries store
key-value pairs and are extremely efficient for lookups.

Control Flow:
Python uses if/elif/else for conditional execution, for loops for iteration over
sequences, while loops for repeated execution, and try/except for error handling.
The for loop in Python iterates over items of any sequence (list, string, etc.)
rather than iterating over arithmetic progressions as in some other languages.

Functions in Python:
Functions are defined using the 'def' keyword. Python supports default arguments,
keyword arguments, *args (variable positional arguments), and **kwargs (variable
keyword arguments). Functions are first-class objects, meaning they can be passed
as arguments, returned from other functions, and assigned to variables.

Python is widely used in web development (Django, Flask), data science (pandas,
NumPy, scikit-learn), artificial intelligence (TensorFlow, PyTorch), automation,
and scripting. It is one of the most popular languages in the world.""")

with open("sample_docs/python_oop.txt", "w") as f:
    f.write("""Object-Oriented Programming (OOP) in Python

Object-Oriented Programming is a paradigm based on the concept of 'objects', which
contain data (attributes) and code (methods). Python has been an object-oriented
language since its inception. Everything in Python is an object.

Classes and Objects:
A class is a blueprint for creating objects. It defines attributes and methods that
the objects will have. You create a class using the 'class' keyword. An object is
an instance of a class. The __init__ method is the constructor, called when a new
object is created.

Example:
class Dog:
    def __init__(self, name, breed):
        self.name = name
        self.breed = breed

    def bark(self):
        return f"{self.name} says Woof!"

The Four Pillars of OOP:

1. Encapsulation: Bundling data and methods within a class, and restricting direct
   access to some components. In Python, we use underscore conventions: _protected
   and __private (name mangling).

2. Inheritance: Creating a new class (child) from an existing class (parent). The
   child inherits attributes and methods from the parent. Python supports multiple
   inheritance, where a class can inherit from multiple parent classes.

3. Polymorphism: The ability to use a common interface for different underlying forms.
   In Python, this is naturally achieved through duck typing — if an object has the
   required method, it can be used regardless of its class.

4. Abstraction: Hiding complex implementation details and showing only the necessary
   features. Python provides abstract base classes (ABC module) for this purpose.

Special Methods (Dunder Methods):
Python uses double-underscore methods like __str__, __repr__, __len__, __add__,
__eq__, etc. to define how objects behave with built-in operations. For example,
defining __str__ controls what happens when you call str() or print() on an object.

Decorators like @property, @staticmethod, and @classmethod provide additional
tools for structuring OOP code in Python.""")

with open("sample_docs/python_errors.txt", "w") as f:
    f.write("""Error Handling and Exceptions in Python

Errors in Python are categorized into two types: syntax errors and exceptions.
Syntax errors occur when the parser detects an incorrect statement. Exceptions
occur during execution and can be handled gracefully.

The Try/Except Block:
The fundamental mechanism for handling exceptions is the try/except block.
Code that might raise an exception is placed in the try block, and the handling
code goes in the except block.

try:
    result = 10 / 0
except ZeroDivisionError:
    print("Cannot divide by zero!")
except TypeError as e:
    print(f"Type error: {e}")
else:
    print("No exception occurred")
finally:
    print("This always runs")

Common Built-in Exceptions:
- ValueError: Raised when a function receives an argument of the right type but inappropriate value.
- TypeError: Raised when an operation is applied to an object of inappropriate type.
- KeyError: Raised when a dictionary key is not found.
- IndexError: Raised when a sequence index is out of range.
- FileNotFoundError: Raised when trying to open a file that doesn't exist.
- AttributeError: Raised when an attribute reference or assignment fails.
- ImportError: Raised when an import statement fails to find the module.

Custom Exceptions:
You can create your own exceptions by inheriting from the Exception class:

class InsufficientFundsError(Exception):
    def __init__(self, balance, amount):
        self.balance = balance
        self.amount = amount
        super().__init__(f"Cannot withdraw {amount}, balance is {balance}")

Best Practices:
1. Catch specific exceptions, not generic ones.
2. Use finally for cleanup operations (closing files, releasing resources).
3. Don't use exceptions for normal control flow.
4. Use context managers (with statement) when working with resources.
5. Log exceptions for debugging purposes.""")

print("✅ Sample documents created in 'sample_docs/' directory:")
for f in os.listdir("sample_docs"):
    size = os.path.getsize(f"sample_docs/{f}")
    print(f"   📄 {f} ({size} bytes)")

In [ ]:
#@title 📥 Load Documents with LangChain

from langchain_community.document_loaders import TextLoader, DirectoryLoader

# Load all files from the directory
dir_loader = DirectoryLoader("sample_docs/", glob="**/*.txt", loader_cls=TextLoader)
all_docs = dir_loader.load()

print(f"📂 Total documents loaded: {len(all_docs)}")
for i, doc in enumerate(all_docs):
    print(f"  📄 Doc {i}: {doc.metadata['source']} ({len(doc.page_content)} chars)")

---
## ✂️ Part 3 — Splitting Documents into Chunks

| Parameter | Description | Typical Value |
|---|---|---|
| `chunk_size` | Maximum characters per chunk | 500–1500 |
| `chunk_overlap` | Shared characters between chunks | 50–200 |

**Overlap** ensures information at chunk boundaries is not lost.

In [ ]:
#@title ✂️ Split Documents into Chunks

from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(all_docs)

print(f"📊 Original documents: {len(all_docs)} → After splitting: {len(chunks)} chunks")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n📦 Chunk {i} | Source: {chunk.metadata['source']} | {len(chunk.page_content)} chars")
    print(f"   {chunk.page_content[:150]}...")

---
## 🧮 Part 4 — Embeddings: Text → Vectors

An **embedding** converts text into a numerical vector that captures its semantic meaning.  
Similar texts → similar vectors → we can search by meaning!

In [ ]:
#@title 🧮 Load the Embedding Model (runs locally, free!)

from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("✅ Embedding model loaded: all-MiniLM-L6-v2 (384 dimensions)")

In [ ]:
#@title 🔬 Hands-On: Explore Cosine Similarity Between Sentences

import numpy as np
from numpy.linalg import norm

sentences = [
    "Python is a programming language",
    "Java is a coding language",
    "I love eating pizza on weekends",
    "Object-oriented programming uses classes and objects",
]

embeddings = embedding_model.embed_documents(sentences)

def cosine_similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

print(f"Each sentence → vector of {len(embeddings[0])} dimensions\n")
print("📐 COSINE SIMILARITY BETWEEN PAIRS:")
print("-" * 60)
for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        sim = cosine_similarity(embeddings[i], embeddings[j])
        emoji = "🟢" if sim > 0.5 else "🟡" if sim > 0.3 else "🔴"
        print(f"  {emoji} {sim:.4f}  \"{sentences[i][:35]}\"  ↔  \"{sentences[j][:35]}\"")

print(f"\n💡 Similar meanings → higher scores!")

---
## 🗄️ Part 5 — Vector Store (ChromaDB)

A **Vector Store** stores embeddings and enables fast similarity search.

In [ ]:
#@title 🗄️ Create the Vector Store & Index Chunks

from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="python_tutorial",
    persist_directory="./chroma_db"
)

print(f"✅ Vector store created with {vectorstore._collection.count()} chunks")

In [ ]:
#@title 🔍 Test Similarity Search

query = "How do I define a class in Python?"
results = vectorstore.similarity_search_with_score(query, k=3)

print(f"🔍 QUERY: \"{query}\"\n")
for i, (doc, score) in enumerate(results):
    print(f"📦 Result {i+1} (distance: {score:.4f}) [{doc.metadata['source']}]")
    print(f"   {doc.page_content[:180]}...\n")

---
## 🔍 Part 6 — The Retriever

In [ ]:
#@title 🔍 Create the Retriever

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

docs = retriever.invoke("What are Python decorators?")
print(f"Retrieved {len(docs)} chunks")
for i, d in enumerate(docs):
    print(f"  📦 {i+1}: {d.page_content[:120]}...")

---
## 🧠 Part 7 — The LLM

In [ ]:
#@title 🧠 Initialize the LLM (OpenAI or HuggingFace fallback)

if USE_OPENAI:
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0.3, max_tokens=1000)
    print("✅ Using OpenAI GPT-3.5-turbo")
else:
    from langchain_community.llms import HuggingFaceHub
    hf_token = getpass("🔑 Enter your HuggingFace token (free at huggingface.co): ")
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token
    llm = HuggingFaceHub(
        repo_id="google/flan-t5-large",
        model_kwargs={"temperature": 0.3, "max_length": 512}
    )
    print("✅ Using HuggingFace flan-t5-large (free)")

response = llm.invoke("What is 2 + 2? Answer in one word.")
print(f"   Quick test: 2+2 = {response}")

---
## 🔗 Part 8 — Building the RAG Chain

The **prompt template** tells the LLM to answer based ONLY on retrieved context.

In [ ]:
#@title 📝 Create the Prompt Template

from langchain.prompts import ChatPromptTemplate

RAG_PROMPT_TEMPLATE = """
You are a helpful teaching assistant for a Python programming course.
Answer the student's question based ONLY on the following context.

Rules:
- If the answer is in the context, provide a clear and detailed explanation.
- If the answer is NOT in the context, say \"I don't have information about that in the course materials.\"
- Include code examples when relevant.
- Be encouraging and educational in tone.

Context:
{context}

Student's Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(RAG_PROMPT_TEMPLATE)
print("✅ Prompt template created with variables: {context} and {question}")

In [ ]:
#@title 🔗 Build the RAG Chain (LCEL)

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    """Combine retrieved documents into a single string."""
    return "\n\n---\n\n".join(
        f"[Source: {doc.metadata.get('source', 'unknown')}]\n{doc.page_content}"
        for doc in docs
    )

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG chain: Question → Retrieve → Format → Prompt → LLM → Answer")

In [ ]:
#@title 🧪 Test the RAG Chain

test_questions = [
    "What are the four pillars of OOP in Python?",
    "How do I handle errors in Python?",
    "What data types does Python support?",
]

for q in test_questions:
    print(f"\n{'='*70}")
    print(f"❓ {q}")
    print(f"{'='*70}")
    print(f"🤖 {rag_chain.invoke(q)}")

In [ ]:
#@title 🔬 Debug: Inspect Every Step of the Pipeline

debug_q = "How do I create a custom exception?"
print(f"🔬 DEBUGGING: \"{debug_q}\"\n")

retrieved = retriever.invoke(debug_q)
print(f"📥 STEP 1 — {len(retrieved)} chunks retrieved:")
for i, doc in enumerate(retrieved):
    print(f"  Chunk {i+1} [{doc.metadata['source']}]: {doc.page_content[:150]}...\n")

ctx = format_docs(retrieved)
print(f"📝 STEP 2 — Formatted context: {ctx[:400]}...\n")

answer = rag_chain.invoke(debug_q)
print(f"🤖 STEP 3 — Final answer:\n{answer}")

---
## 💬 Part 9 — Conversation Memory

In [ ]:
#@title 💬 Build the Conversational RAG Chain

from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(
    k=5, memory_key="chat_history",
    return_messages=True, output_key="answer"
)

conversational_rag = ConversationalRetrievalChain.from_llm(
    llm=llm, retriever=retriever,
    memory=memory, return_source_documents=True, verbose=False
)

print("✅ Conversational RAG chain built with memory (last 5 exchanges)")

In [ ]:
#@title 🧪 Test Follow-Up Questions

conversation = [
    "What is inheritance in Python?",
    "Can you show me an example?",
    "What about multiple inheritance?",
    "How does error handling work?",
    "What are the best practices for that?",
]

for q in conversation:
    print(f"\n{'='*70}\n👤 STUDENT: {q}\n{'='*70}")
    result = conversational_rag.invoke({"question": q})
    print(f"🤖 {result['answer']}")
    print(f"📚 Sources: {[d.metadata['source'] for d in result['source_documents']]}")

---
## 🖥️ Part 10 — Interactive Chat Interface

In [ ]:
#@title 🖥️ Interactive Chat (type 'quit' to exit)

memory.clear()
print("🤖 Python Tutorial Chatbot")
print("="*50)
print("Type 'quit' to exit, 'history' to see chat log")
print("="*50)

while True:
    question = input("\n👤 You: ").strip()
    if not question: continue
    if question.lower() == 'quit':
        print("👋 Goodbye!"); break
    if question.lower() == 'history':
        for msg in memory.chat_memory.messages:
            role = "👤" if msg.type == "human" else "🤖"
            print(f"  {role} {msg.content[:120]}")
        continue
    try:
        result = conversational_rag.invoke({"question": question})
        print(f"\n🤖 {result['answer']}")
        print(f"   📚 {set(d.metadata['source'] for d in result['source_documents'])}")
    except Exception as e:
        print(f"❌ Error: {e}")

---
## 🎓 Part 11 — Summary

| Step | Component | LangChain Class | Purpose |
|---|---|---|---|
| 1 | Document Loader | `TextLoader`, `PyPDFLoader` | Load raw documents |
| 2 | Text Splitter | `RecursiveCharacterTextSplitter` | Break docs into chunks |
| 3 | Embedding Model | `HuggingFaceEmbeddings` | Convert text → vectors |
| 4 | Vector Store | `Chroma` | Store and search vectors |
| 5 | Retriever | `vectorstore.as_retriever()` | Find relevant chunks |
| 6 | Prompt Template | `ChatPromptTemplate` | Structure the LLM input |
| 7 | LLM | `ChatOpenAI` / `HuggingFaceHub` | Generate answers |
| 8 | Memory | `ConversationBufferWindowMemory` | Track conversation |
| 9 | Chain | `ConversationalRetrievalChain` | Connect everything |

---

## 🚀 Part 12 — Exercises

1. **Change the Documents** — Use your own PDFs or web pages  
2. **Tune Parameters** — Try chunk_size 200 vs 1000, overlap 0 vs 200, k=1 vs k=10  
3. **Add Source Citations** — Modify the prompt to cite file names  
4. **Try Different Embeddings** — Use `all-mpnet-base-v2` or `paraphrase-multilingual-MiniLM-L12-v2`  
5. **Add a Web Loader** — Load a Wikipedia page into your knowledge base  

---

---

## 📤 Part 13 — Upload YOUR Own File & Chat With It!

Upload **any PDF or text file** from your computer and build a RAG chatbot on top of it.

```
Your File (PDF/TXT) → Upload → Load → Split → Embed → Index → Chat!
```

In [ ]:
#@title 📤 Step 13.1 — Upload Your File

from google.colab import files

print("📤 Select a file from your computer (PDF or TXT):")
uploaded = files.upload()

if not uploaded:
    print("❌ No file uploaded. Run this cell again.")
else:
    uploaded_filename = list(uploaded.keys())[0]
    print(f"\n✅ Uploaded: {uploaded_filename} ({len(uploaded[uploaded_filename]):,} bytes)")

In [ ]:
#@title 📥 Step 13.2 — Load, Split & Index Your Document

from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

ext = os.path.splitext(uploaded_filename)[1].lower()

if ext == ".pdf":
    loader = PyPDFLoader(uploaded_filename)
    print(f"📄 Loading PDF...")
else:
    loader = TextLoader(uploaded_filename, encoding="utf-8")
    print(f"📄 Loading text file...")

user_docs = loader.load()
print(f"   {len(user_docs)} section(s), {sum(len(d.page_content) for d in user_docs):,} total characters")

user_chunks = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=100
).split_documents(user_docs)
print(f"✂️ Split into {len(user_chunks)} chunks")

user_vectorstore = Chroma.from_documents(
    documents=user_chunks, embedding=embedding_model, collection_name="user_upload"
)
user_retriever = user_vectorstore.as_retriever(search_kwargs={"k": 4})

print(f"🗄️ Indexed! Ready to chat with your document.")

In [ ]:
#@title 💬 Step 13.3 — Build Chain for Your Document

from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferWindowMemory

user_memory = ConversationBufferWindowMemory(
    k=5, memory_key="chat_history",
    return_messages=True, output_key="answer"
)

user_chain = ConversationalRetrievalChain.from_llm(
    llm=llm, retriever=user_retriever,
    memory=user_memory, return_source_documents=True
)

print(f"✅ Chain ready for: {uploaded_filename}")

In [ ]:
#@title 🗣️ Step 13.4 — Chat With Your Document! (type 'quit' to exit)

print(f"🤖 Chat with: {uploaded_filename}")
print("="*55)
print("Type 'quit' to exit, 'sources' to see retrieved chunks")
print("="*55)

last_result = None
while True:
    q = input("\n👤 You: ").strip()
    if not q: continue
    if q.lower() == 'quit':
        print("👋 Goodbye!"); break
    if q.lower() == 'sources' and last_result:
        print("\n📚 Source chunks used:")
        for i, doc in enumerate(last_result['source_documents']):
            pg = doc.metadata.get('page', '?')
            print(f"  📦 Chunk {i+1} (page {pg}): {doc.page_content[:200]}...")
        continue
    try:
        last_result = user_chain.invoke({"question": q})
        print(f"\n🤖 {last_result['answer']}")
        print(f"   💡 ({len(last_result['source_documents'])} chunks used — type 'sources' to inspect)")
    except Exception as e:
        print(f"❌ Error: {e}")

---

## 📊 Part 14 — Evaluating RAG with RAGAS

### What is RAGAS?
**RAGAS** (Retrieval Augmented Generation Assessment) evaluates your RAG pipeline:

| Metric | Measures | Ideal |
|---|---|---|
| **Faithfulness** | Is the answer grounded in retrieved context? (no hallucinations) | 1.0 |
| **Answer Relevancy** | Is the answer relevant to the question? | 1.0 |
| **Context Precision** | Are retrieved chunks relevant? | 1.0 |
| **Context Recall** | Did we retrieve all necessary info? | 1.0 |

⚠️ RAGAS uses OpenAI internally for evaluation — you need an API key for this section.

In [ ]:
#@title 📝 Step 14.1 — Create a Ground-Truth Test Dataset

# A proper evaluation needs:
#  - questions
#  - known correct answers (ground truth)
#  - the RAG-generated answers
#  - the retrieved contexts

eval_questions = [
    "What are the four pillars of Object-Oriented Programming?",
    "How do you handle exceptions in Python?",
    "What is duck typing in Python?",
    "What is the difference between a list and a tuple in Python?",
    "How do you define a custom exception in Python?",
]

eval_ground_truths = [
    "The four pillars of OOP are Encapsulation, Inheritance, Polymorphism, and Abstraction.",
    "Exceptions are handled using try/except blocks. Code that might raise an exception goes in try, handling code in except. You can also use else and finally.",
    "Duck typing means that if an object has the required method, it can be used regardless of its class. It is how Python naturally achieves polymorphism.",
    "Lists are mutable ordered sequences, while tuples are immutable. Both store multiple items but tuples cannot be changed after creation.",
    "You create custom exceptions by inheriting from the Exception class and defining an __init__ method that calls super().__init__() with your error message.",
]

print(f"✅ Test dataset: {len(eval_questions)} question-answer pairs")
print(f"   Sample Q: {eval_questions[0]}")
print(f"   Sample A: {eval_ground_truths[0]}")

In [ ]:
#@title 🔄 Step 14.2 — Generate RAG Answers & Collect Contexts

eval_answers = []
eval_contexts = []

print("🔄 Generating answers for evaluation...\n")

for i, question in enumerate(eval_questions):
    answer = rag_chain.invoke(question)
    eval_answers.append(answer)

    retrieved = retriever.invoke(question)
    eval_contexts.append([doc.page_content for doc in retrieved])

    print(f"  ✅ Q{i+1}: {question[:50]}...")
    print(f"     → {answer[:80]}...\n")

print(f"✅ All {len(eval_questions)} answers generated!")

In [ ]:
#@title 📊 Step 14.3 — Run RAGAS Evaluation

from datasets import Dataset

if not USE_OPENAI:
    print("⚠️ RAGAS evaluation requires an OpenAI API key.")
    print("   Read the code to understand how it works, then set your key and re-run!")
else:
    from ragas import evaluate
    from ragas.metrics import (
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    )

    eval_dataset = Dataset.from_dict({
        "question": eval_questions,
        "answer": eval_answers,
        "contexts": eval_contexts,
        "ground_truth": eval_ground_truths,
    })

    print("📊 Running RAGAS evaluation (1-2 min)...\n")

    results = evaluate(
        eval_dataset,
        metrics=[faithfulness, answer_relevancy, context_precision, context_recall]
    )

    print("=" * 60)
    print("📊 RAGAS RESULTS")
    print("=" * 60)
    for metric, score in results.items():
        bar = "█" * int(score * 20) + "░" * (20 - int(score * 20))
        emoji = "🟢" if score >= 0.8 else "🟡" if score >= 0.6 else "🔴"
        print(f"  {emoji} {metric:<25} {bar} {score:.4f}")
    print("=" * 60)

In [ ]:
#@title 📋 Step 14.4 — Per-Question Breakdown

if USE_OPENAI:
    df = results.to_pandas()
    print("📋 PER-QUESTION SCORES\n")
    for idx, row in df.iterrows():
        print(f"❓ Q{idx+1}: {row['question']}")
        print(f"   🤖 {str(row['answer'])[:100]}...")
        for m in ['faithfulness','answer_relevancy','context_precision','context_recall']:
            v = row.get(m, None)
            if v is not None and isinstance(v, float):
                print(f"   📏 {m}: {v:.3f}")
        print("-"*60)
    print("\n📊 Full DataFrame:")
    display(df)
else:
    print("⚠️ Skipped — requires OpenAI API key.")

### 🧠 Interpreting & Improving Scores

| Score | Meaning | Action |
|---|---|---|
| **0.8 – 1.0** 🟢 | Excellent | No action needed |
| **0.6 – 0.8** 🟡 | Acceptable | Room for improvement |
| **0.0 – 0.6** 🔴 | Poor | Needs tuning |

| Low Metric | Problem | Fix |
|---|---|---|
| Faithfulness | LLM hallucinating | Stronger "only use context" prompt |
| Answer Relevancy | Off-topic answers | Better prompt, lower temperature |
| Context Precision | Irrelevant chunks retrieved | Smaller chunk size, better embeddings |
| Context Recall | Missing relevant info | Increase k, more overlap |

---

---

## 🌐 Part 15 — Gradio Web App

Let's turn our RAG chatbot into a **real web application** with:
- Chat interface with message history
- File upload to add new documents
- Adjustable chunk_size, overlap, and k
- Source chunk display
- A **public URL** anyone can access!

In [ ]:
#@title 🌐 Launch the Gradio RAG Chatbot App

import gradio as gr
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import shutil

# ── Global State ──
app_vectorstore = None
app_retriever = None
app_rag_chain = None
current_doc_name = "Python Tutorial (default)"

# ── Helper: build/rebuild RAG chain ──
def build_chain(docs, chunk_size=500, chunk_overlap=100, k=3):
    global app_vectorstore, app_retriever, app_rag_chain

    split_chunks = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    ).split_documents(docs)

    db_path = "./gradio_chroma_db"
    if os.path.exists(db_path):
        shutil.rmtree(db_path)

    app_vectorstore = Chroma.from_documents(
        documents=split_chunks, embedding=embedding_model,
        collection_name="gradio_rag", persist_directory=db_path
    )
    app_retriever = app_vectorstore.as_retriever(search_kwargs={"k": k})

    rag_prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Answer ONLY from the context below.
If the answer is not in the context, say "I don't have information about that."

Context:
{context}

Question: {question}

Answer:""")

    def fmt(docs):
        return "\n\n---\n\n".join(d.page_content for d in docs)

    app_rag_chain = (
        {"context": app_retriever | fmt, "question": RunnablePassthrough()}
        | rag_prompt | llm | StrOutputParser()
    )
    return len(split_chunks)

# Initialize with default docs
n_chunks = build_chain(all_docs)

# ── Callbacks ──
def upload_file(file, chunk_size, chunk_overlap, k):
    global current_doc_name
    if file is None:
        return "⚠️ No file selected.", []
    filepath = file.name
    ext = os.path.splitext(filepath)[1].lower()
    try:
        loader = PyPDFLoader(filepath) if ext == ".pdf" else TextLoader(filepath, encoding="utf-8")
        docs = loader.load()
        n = build_chain(docs, int(chunk_size), int(chunk_overlap), int(k))
        current_doc_name = os.path.basename(filepath)
        return f"✅ **{current_doc_name}** loaded! {len(docs)} section(s), {n} chunks (size={int(chunk_size)}, overlap={int(chunk_overlap)}, k={int(k)})", []
    except Exception as e:
        return f"❌ Error: {e}", []

def chat(user_message, history):
    if not user_message.strip():
        return history, ""
    if app_rag_chain is None:
        history.append((user_message, "⚠️ No document loaded."))
        return history, ""
    try:
        answer = app_rag_chain.invoke(user_message)
        retrieved = app_retriever.invoke(user_message)
        sources = "\n\n".join(
            f"📦 **Chunk {i+1}**: {d.page_content[:200]}..."
            for i, d in enumerate(retrieved)
        )
        full = f"{answer}\n\n---\n📚 **Sources:**\n{sources}"
        history.append((user_message, full))
    except Exception as e:
        history.append((user_message, f"❌ {e}"))
    return history, ""

# ── Build UI ──
with gr.Blocks(title="RAG Chatbot", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 RAG Chatbot — LangChain + Gradio\n**Upload a document and ask questions!** — Prof.ssa Flora Amato, UNINA DIETI")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### ⚙️ Settings")
            file_upload = gr.File(label="📤 Upload (PDF/TXT)", file_types=[".pdf",".txt",".md",".py"])
            chunk_sz = gr.Slider(200, 2000, 500, step=100, label="Chunk Size")
            overlap = gr.Slider(0, 400, 100, step=50, label="Chunk Overlap")
            k_val = gr.Slider(1, 10, 3, step=1, label="k (chunks to retrieve)")
            load_btn = gr.Button("📥 Load Document", variant="primary")
            status = gr.Markdown(f"📄 Current: **{current_doc_name}** ({n_chunks} chunks)")

        with gr.Column(scale=2):
            gr.Markdown("### 💬 Chat")
            chatbot = gr.Chatbot(height=450, show_copy_button=True)
            with gr.Row():
                msg = gr.Textbox(placeholder="Ask a question...", show_label=False, scale=4)
                send = gr.Button("Send 📤", variant="primary", scale=1)
            clear = gr.Button("🗑️ Clear Chat")

    load_btn.click(upload_file, [file_upload, chunk_sz, overlap, k_val], [status, chatbot])
    send.click(chat, [msg, chatbot], [chatbot, msg])
    msg.submit(chat, [msg, chatbot], [chatbot, msg])
    clear.click(lambda: [], outputs=[chatbot])

print("🚀 Launching Gradio app...")
demo.launch(share=True, debug=False, show_error=True)

### 💡 Things to Try with the Gradio App

1. Upload a PDF of a textbook chapter and ask questions  
2. Change chunk_size and see how answer quality changes  
3. Try k=1 vs k=10 — does more context always help?  
4. Share the public URL with a classmate!  

---

## 📚 Further Reading

- [LangChain Documentation](https://python.langchain.com/docs/)
- [RAG Tutorial — LangChain](https://python.langchain.com/docs/tutorials/rag/)
- [ChromaDB Documentation](https://docs.trychroma.com/)
- [RAGAS Documentation](https://docs.ragas.io/)
- [Gradio Documentation](https://www.gradio.app/docs/)
- [HuggingFace Sentence Transformers](https://www.sbert.net/)

### Advanced Topics
- **Hybrid Search**: Combine vector search + BM25 keyword search
- **Re-ranking**: Cross-encoders to re-rank retrieved results
- **Agentic RAG**: LLM decides when/how to retrieve
- **Multi-modal RAG**: Images, tables, code alongside text

---

**🎓 End of Tutorial — Prof.ssa Flora Amato, DIETI, UNINA**  
**Happy coding! 🐍**